## X (Twitter)

#### Useful links:
- [How to set an account](https://docs.x.com/x-api/getting-started/getting-access)
- [X (Twitter) API Documentation](https://developer.x.com/en/docs/x-api)
- [Tweepy - Python Client](https://docs.tweepy.org/en/stable/)


In [1]:
# requirements
# %pip install pandas python-dotenv tweepy

In [2]:
import os
import json
import pandas as pd
import tweepy
import requests
from dotenv import load_dotenv
from datetime import datetime

In [3]:
# environment variables
load_dotenv()
TWITTER_BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN")

In [ ]:
# API connection
api = tweepy.Client(bearer_token=TWITTER_BEARER_TOKEN)


# Pagination example
def paginate_recent_tweets(query, limit):
    all_tweets = []
    for status in tweepy.Cursor(
        api.search_recent_tweets, query, count=100
    ).items(limit):
        all_tweets.append(status)
    return all_tweets

### SPECIAL CRITERIA

**SC1: Does the platform offer an API for collecting public user-generated content data?**

*This item verifies whether the platform provides an API with at least one endpoint for programmatically extracting public user-generated content to the users’ infrastructure. Public user-generated content is defined here as any publicly visible publication accessible by any platform user. The assessment should verify that the endpoint allows retrieval and storage of this content without requiring privileged or internal access beyond standard developer registration.*

In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.


# Using the Tweepy Client gave me some issues, so I'm using requests directly
# for now. Every time I try to use Tweepy, I get TooManyRequests errors. Maybe
# it's because I'm using a free account.

# Considering that a publication is a tweet, I'm searching for tweets that match a specific query.


def search_all_tweets(query, max_results=10, next_token=None):
    # This endpoint is only available for the Pro and Enterpise accounts.
    # It returns tweets from the full archive that match a specific query.

    params = {"query": query, "max_results": max_results}
    if next_token:
        params["next_token"] = next_token

    tweets = requests.get(
        "https://api.x.com/2/tweets/search/all",
        params=params,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
    ).json()
    return tweets


def search_recent_tweets(query, max_results=1, next_token=None):
    # This endpoint returns the most recent in the last 7 days. Max 100 per request.
    # To get more, we would need to paginate using the next_token that comes in the response.

    # The header is set above with the bearer token. You can get the bearer token from your Twitter developer account.

    # To get a more detailed response, you can see the documentation here:
    # https://docs.x.com/x-api/posts/search-recent-posts#search-recent-posts

    # Set up the query parameters
    params = {"query": query, "max_results": max_results}
    if next_token:
        params["next_token"] = next_token

    tweets = requests.get(
        "https://api.x.com/2/tweets/search/recent",
        params=params,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
    ).json()
    return tweets


result = search_recent_tweets("from:nytimes", max_results=10)
result_all = search_all_tweets("from:nytimes", max_results=10)
print(f"recent tweets response: {json.dumps(result, indent=2)}")
print(
    f"\nAll tweets response (should fail for free accounts): {json.dumps(result_all, indent=2)}"
)

recent tweets response: {
  "data": [
    {
      "text": "Breaking News: President Trump is considering an overhaul to the U.S. refugee program that would favor mostly white people, documents show. https://t.co/wb2xBaF2UX",
      "id": "1978528187735798247",
      "edit_history_tweet_ids": [
        "1978528187735798247"
      ]
    },
    {
      "text": "Breaking News: The Trump administration secretly authorized the CIA to conduct covert action in Venezuela, according to U.S. officials. https://t.co/PoYVsP1SX2",
      "id": "1978524809513295921",
      "edit_history_tweet_ids": [
        "1978524809513295921"
      ]
    },
    {
      "text": "Breaking News: A man has been charged with raping and killing a teenager on Long Island, New York, in a 1984 cold case after the authorities analyzed DNA evidence. https://t.co/fNSwxoXSiJ",
      "id": "1978518456656891983",
      "edit_history_tweet_ids": [
        "1978518456656891983"
      ]
    },
    {
      "text": "From @TheAthletic:

### ACCESSIBILITY


**OC1: Does the platform offer any type of access to non-public data (e.g., private groups, not direct messages) to approved researchers?**

*This item verifies whether, beyond the retrieval and extraction of public user-generated content data, the API permits the extraction of any data from non-public content, such as posts and comments in private groups or discussion forums, subject to the implementation of adequate data hashing measures and specific researcher approval. The assessment should confirm that the API provides such access measures, either through specific endpoints or other controlled access mechanisms.*

In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# It does not offer access to non-public data, such as deleted tweets or tweets from private accounts.
# It does not provide any kind of research or academic access to Twitter data.

**OC2: Can the requested data be extracted directly from the platform’s API response?**

*This item verifies whether the API returns structured data directly in its response, rather than only providing a redirect link to the data. Audiovisual media (e.g., images, videos, and audio) are excluded from this assessment. The assessment should check sample API responses to confirm that the requested public data is included in the returned payload itself.*


In [ ]:
def search_recent_tweets_with_fields(
    query, max_results=1, next_token=None, twitter_fields=[]
):
    # This endpoint returns the most recent in the last 7 days. Max 100 per request.
    # To get more, we would need to paginate using the next_token that comes in the response.

    # Here we can specify which fields we want to get in the response.

    # The header is set above with the bearer token. You can get the bearer token from your Twitter developer account.

    # To get a more detailed response, you can see the documentation here:
    # https://docs.x.com/x-api/posts/search-recent-posts#search-recent-posts

    # Set up the query parameters
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": ",".join(twitter_fields),
    }
    if next_token:
        params["next_token"] = next_token

    tweets = requests.get(
        "https://api.x.com/2/tweets/search/recent",
        params=params,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
    )
    return tweets

In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# Yes it is possible to specify which fields to return in the response.


available_fields = [
    "article",
    "attachments",
    "author_id",
    "card_uri",
    "community_id",
    "context_annotations",
    "conversation_id",
    "created_at",
    "display_text_range",
    "edit_controls",
    "edit_history_tweet_ids",
    "entities",
    "geo",
    "id",
    "in_reply_to_user_id",
    "lang",
    "media_metadata",
    "non_public_metrics",
    "note_tweet",
    "organic_metrics",
    "possibly_sensitive",
    "promoted_metrics",
    "public_metrics",
    "referenced_tweets",
    "reply_settings",
    "scopes",
    "source",
    "suggested_source_links",
    "text",
    "withheld",
]
non_available_fields = [
    "non_public_metrics",
    "promoted_metrics",
    "organic_metrics",
]

fields = set(available_fields) - set(non_available_fields)
fields = list(fields)  # converting back to list

result = search_recent_tweets_with_fields(
    "from:nytimes", max_results=10, twitter_fields=fields
)
print(
    f"recent tweets response with all fields: {json.dumps(result.json(), indent=2)}"
)

recent tweets response with all fields: {
  "data": [
    {
      "entities": {
        "annotations": [
          {
            "start": 104,
            "end": 112,
            "probability": 0.9622,
            "type": "Place",
            "normalized_text": "Argentina"
          },
          {
            "start": 124,
            "end": 125,
            "probability": 0.9312,
            "type": "Place",
            "normalized_text": "US"
          }
        ],
        "mentions": [
          {
            "start": 3,
            "end": 14,
            "username": "arappeport",
            "id": "14480378"
          },
          {
            "start": 23,
            "end": 35,
            "username": "colbyLsmith",
            "id": "1176918506"
          },
          {
            "start": 46,
            "end": 62,
            "username": "SecScottBessent",
            "id": "1889019333960998912"
          }
        ]
      },
      "conversation_id": "1979235854317920716",
  

**OC4: Does the platform’s API offer an endpoint for extracting data from an individual publication?**

*This item verifies whether it is possible to collect data from a specific public post on the platform using a unique identifier, rather than relying on search terms or other filters. The assessment should review the API documentation and test available endpoints to confirm that an individual publication can be retrieved directly by its unique identifier.*


In [ ]:
# Use this cell to develop an example where a request for a specific post is made (by its id).
# Please leave the result as the output of this cell.
tweet_id = 1979235854317920716  # SET THE VALUE HERE


def get_tweet_by_id(tweet_id):
    url = f"https://api.x.com/2/tweets/{tweet_id}"
    # params = {
    #     "tweet.fields": ["created_at", "author_id", "text"]
    # }
    params = {}
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
        params=params,
    )
    return response


result = get_tweet_by_id(tweet_id)
print(f"tweet by id response: {json.dumps(result.json(), indent=2)}")

tweet by id response: {
  "data": {
    "text": "RT @arappeport: New w/ @colbyLsmith: Treasury @SecScottBessent stakes credibility and taxpayer money on Argentina bet.\n\nThe US bailout of a\u2026",
    "id": "1979235854317920716",
    "edit_history_tweet_ids": [
      "1979235854317920716"
    ]
  }
}


**OC5: Does the platform’s API offer an endpoint for extracting data from an individual author?**

*This item verifies whether it is possible to collect data from public posts made by a specific author, using their username or unique identifier. The assessment should review the API documentation and test relevant endpoints to confirm that data can be retrieved directly for an individual author.*


In [ ]:
# Use this cell to develop an example where a request for a specific user is made.
# Please leave the result as the output of this cell.
user = 807095  # SET THE VALUE HERE


def get_user_tweets(user: int):
    url = f"https://api.x.com/2/users/{user}/tweets"
    params = {"tweet.fields": ",".join(["created_at", "author_id", "text"])}
    # params= {}
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
        params=params,
    )
    return response


result = get_user_tweets(user)
print(f"user tweets response: {json.dumps(result.json(), indent=2)}")

user tweets response: {
  "data": [
    {
      "edit_history_tweet_ids": [
        "2012156353842643198"
      ],
      "author_id": "807095",
      "text": "Breaking News: The CIA director met with Venezuela\u2019s acting leader in Caracas, a bolstering of the government that could be seen as a snub to the opposition. https://t.co/4MfzYQglk5",
      "id": "2012156353842643198",
      "created_at": "2026-01-16T13:33:53.000Z"
    },
    {
      "edit_history_tweet_ids": [
        "2012090426325430561"
      ],
      "author_id": "807095",
      "text": "Breaking News: Yoon Suk Yeol, South Korea\u2019s ousted leader, got five years in prison in the first sentence from his martial law trials. https://t.co/zxa7dJCDQJ",
      "id": "2012090426325430561",
      "created_at": "2026-01-16T09:11:55.000Z"
    },
    {
      "edit_history_tweet_ids": [
        "2012021962701488624"
      ],
      "author_id": "807095",
      "text": "RT @Lattif: Iran appeared to backpedal on previous threats to 

**OC6: Does the platform’s API provide an endpoint for extracting data based on search terms?**

*This item verifies whether public user-generated content can be collected via individual or combined search terms, enabling the creation of datasets of posts mentioning those terms. The assessment should test search-related endpoints to confirm that queries using keywords return relevant public posts.*


In [ ]:
# Use this cell to develop an example where a request for posts is made using a search term.
# Please leave the result as the output of this cell.
# search_term = "globo" # SET THE VALUE HERE

# # Yes, it is possible (Same function as OC2)

# result = search_recent_tweets_with_fields(search_term, max_results=10)
# print(f"recent tweets response for search term '{search_term}': {json.dumps(result.json(), indent=2)}")

search_terms = "globo OR globolixo"

result_2 = search_recent_tweets_with_fields(search_terms, max_results=10)
print(
    f"recent tweets response for search term '{search_terms}': {json.dumps(result_2.json(), indent=2)}"
)

recent tweets response for search term 'globo OR globolixo': {
  "data": [
    {
      "text": "RT @bellanna: Vcs s\u00e3o muito caras de pau. PUTA QUE ME PARIU. N\u00e3o \u00e9 a toa que s\u00e3o afiliados da globo. https://t.co/mrFUA7Zvee",
      "edit_history_tweet_ids": [
        "1980283894055227402"
      ],
      "id": "1980283894055227402"
    },
    {
      "text": "Mentira exceto bumerangue (mt antigo) e globo (muito recente)",
      "edit_history_tweet_ids": [
        "1980283890414547070"
      ],
      "id": "1980283890414547070"
    },
    {
      "text": "@carlosaviana A ESCR@T@ EXTREMA-DIREITA ESTARIA COM MEDO?\ud83e\udd23\ud83e\udd23\n\"DEPUTADA @Biakicis (PL-DF) E SENADOR @IzalciLucas (PL-DF) \nVOLTARAM ATR\u00c1S EM SEUS PEDIDOS PARA CONVOCAR O ADVOGADO E QUEBRAR SEU SIGILO FISCAL E BANC\u00c1RIO. OS DOIS N\u00c3O JUSTIFICARAM A SOLICITA\u00c7\u00c3O DE RETIRADA.\"\nhttps://t.co/xbM6QSXtjp",
      "edit_history_tweet_ids": [
        "1980283888078348764"
      ],
   

**OC7: Does the API use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are returned in a locale-neutral format, or whether relevant locale metadata is included when neutrality is not possible. The assessment should review the API documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.


# Partially, some info does appears to be locale-neutral, like "created_at", but most fields are not consistent and cannot be checked for locale-neutrality.
# There are no specific parameters to enforce locale-neutrality in the response and many fields that should be relevant for this purpose, even if available, do not show in the response.
# e.g. "place.fields": ["country", "full_name"] does not return any data in the response, even if the tweet has geo info.
# e.g. "lang" field is present, but it is inferred from the tweet content, not from any locale-neutral metadata.
# Timestamps, such as "created_at", are in UTC, which is locale-neutral. Did not find any timezone info in any other tweet.

params = {
    "query": "from:nytimes",
    "max_results": 10,
    "twitter_fields": ["lang", "created_at", "geo"],
    "user_fields": ["location"],
    "place_fields": ["country", "full_name"],
}


def check_locale_neutrality(
    query, max_results=10, twitter_fields=[], user_fields=[], place_fields=[]
):
    url = "https://api.x.com/2/tweets/search/recent"
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": ",".join(twitter_fields),
        "user.fields": ",".join(user_fields),
        "place.fields": ",".join(place_fields),
    }

    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
        params=params,
    )
    return response


result = check_locale_neutrality(**params)
print(
    f"locale neutrality check response: {json.dumps(result.json(), indent=2)}"
)

locale neutrality check response: {
  "data": [
    {
      "lang": "en",
      "text": "From @TheAthletic: For the first time since 1991, a five-day sumo wrestling tournament was held outside of Japan. It was a sight to behold. A sell-out crowd, many of whom had travelled from all over Europe to witness the spectacle, were engrossed. https://t.co/Ap3A0y0Bp5",
      "created_at": "2025-10-20T18:44:57.000Z",
      "edit_history_tweet_ids": [
        "1980344505153818930"
      ],
      "id": "1980344505153818930"
    },
    {
      "lang": "en",
      "text": "From @TheAthletic: On Friday, Shohei Ohtani took his team to the World Series in what some writers called \"the greatest performance ever.\" But is it the greatest individual performance in all of sports? Our experts took a look. https://t.co/Vs06VTDvQh https://t.co/7eR9NIbiUv",
      "created_at": "2025-10-20T18:04:03.000Z",
      "edit_history_tweet_ids": [
        "1980334211727855822"
      ],
      "id": "1980334211727855822"

### COMPLIANCE

**OC15: Does the platform provide a way to label content that has been generated with artificial intelligence?**

*This item verifies whether the platform automatically flags, or allows users to flag, AI-generated content, and whether this information is given in the API response. The assessment should review the documentation and test API outputs to confirm that these flags are included in the extracted data.* 


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# There is no flags that indicates that a content was AI-generated.
# There are no parameters to enable a user to flag AI-generated content on Twitter.
#
# To check this, I reviewed the Twitter API documentation through all available resources and endpoints.
# Also checked with the Docs AI Chat Support Bot and community forum pages.
# Source: https://docs.x.com/x-api/
# Even the Docs AI Chat Support Bot only find AI mentions in the Community Page AI Writer program.

### COMPLETENESS

**OC16: Can data from a publication’s comments be extracted using the platform’s API?**

*This item verifies whether comment data, including their content, can be extracted when available on the platform, either together with publication data or with a dedicated endpoint. The assessment should test relevant endpoints to confirm that comments are retrievable as structured data. This item does not apply to platforms that do not have commenting features.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# Yes, it is possible to retrieve publication's comments (replies to a tweet).
# The endpoint used is the same as for searching recent tweets, with a specific query to filter replies.
# The query used is "in_reply_to_tweet_id:{tweet_id}" where tweet_id is the id of the tweet for which we want to get the replies.

result = search_recent_tweets_with_fields(
    "in_reply_to_tweet_id:1980950885174738961",
    max_results=10,
    twitter_fields=["id", "text", "created_at", "lang"],
)
print(
    f"recent tweets in reply to tweet id 1980950885174738961: {json.dumps(result.json(), indent=2)}"
)

recent tweets in reply to tweet id 1980950885174738961: {
  "data": [
    {
      "lang": "pt",
      "edit_history_tweet_ids": [
        "1981057936517714052"
      ],
      "id": "1981057936517714052",
      "created_at": "2025-10-22T17:59:52.000Z",
      "text": "@ivanvieira_5 E Lula na rua tamb\u00e9m..."
    },
    {
      "lang": "pt",
      "edit_history_tweet_ids": [
        "1981057408714887287"
      ],
      "id": "1981057408714887287",
      "created_at": "2025-10-22T17:57:47.000Z",
      "text": "@ivanvieira_5 ESTADO DEMOCR\u00c1TICO DE DREITO: VAMOS EXIGIR O VOTO IMPRESSO AUDIT\u00c1VEL COM VOTO IMPRESSO AUDIT\u00c1VEL. \u00c9 a \u00fanica e leg\u00edtima alternativa para termos elei\u00e7\u00f5es limpas, sem falcatrua e sem malandragens do TSE e tirar os comunistas e nazista do poder, COM URNAS IGUAIS DAS DA VENEZUELA."
    },
    {
      "lang": "pt",
      "edit_history_tweet_ids": [
        "1981056182283309392"
      ],
      "id": "1981056182283309392",
      "creat

**OC17: Can data from temporary content be extracted through the platform’s API?**

*This item verifies whether the platform’s API provides at least one endpoint for collecting data from temporary publications (e.g., stories, ephemeral messages). The assessment should test endpoints to confirm whether this type of short-lived content can be retrieved as structured data before it expires. This item does not apply to platforms that do not have temporary content features.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# Twitter Spaces are live audio conversations on Twitter. They allow users to host and join live audio discussions.
params = {
    "space_id": "1MnxnMDeQLeJO",  # SET THE VALUE HERE
    "space_fields": [
        "created_at",
        "creator_id",
        "ended_at",
        "host_ids",
        "invited_user_ids",
        "participant_count",
        "speaker_ids",
        "state",
        "started_at",
        "title",
        "updated_at",
    ],
    "media_fields": ["url", "preview_image_url", "public_metrics", "type"],
}


def get_spaces(space_id, space_fields=[], media_fields=[]):
    url = f"https://api.x.com/2/spaces/{space_id}"
    params = {
        "space.fields": ",".join(space_fields),
        # "media.fields": ",".join(media_fields),
    }

    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
        params=params,
    )
    return response


result = get_spaces(**params)
print(f"spaces response: {json.dumps(result.json(), indent=2)}")

spaces response: {
  "data": {
    "created_at": "2024-06-24T16:20:08.000Z",
    "updated_at": "2024-06-29T03:23:30.000Z",
    "invited_user_ids": [
      "19737700",
      "1306359926164516864",
      "926868226654404608"
    ],
    "started_at": "2024-06-29T01:01:16.000Z",
    "id": "1MnxnMDeQLeJO",
    "title": "\ud83d\udea8@XSPACES Q & A FEEDBACK W/ \ud835\udd4f LEAD MEDIA ENGINEER @marmars",
    "creator_id": "1585809424135995392",
    "state": "ended",
    "ended_at": "2024-06-29T03:21:43.000Z",
    "participant_count": 46383,
    "speaker_ids": [
      "1306359926164516864",
      "1522405142888255488",
      "926868226654404608",
      "1176026136117166080",
      "1517929379879436288",
      "1524504457853030401",
      "153438388",
      "14437705",
      "50660218",
      "26335586",
      "134910622",
      "339182018",
      "884299568837349377",
      "17050482"
    ],
    "host_ids": [
      "1585809424135995392",
      "1176026136117166080",
      "1522405142888255488"


**OC18: Can historical data be extracted through the platform’s API?**

*This item verifies whether the API provides endpoints that allow for a specified time range, going back more than one year from the time the request is made, to collect public user-generated content data. The assessment should review test endpoints to confirm that historical data more than 12 months prior to the analysis can be retrieved.*

In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# For the free tier of the X API (v2), it is not possible to search tweets by a specific date range.
# The "search/recent" endpoint only allows searching tweets from the last 7 days.


def search_recent_tweets_historical(
    query,
    max_results=1,
    next_token=None,
    twitter_fields=[],
    start_time=None,
    end_time=None,
):
    # This endpoint returns the most recent in the last 7 days. Max 100 per request.
    # To get more, we would need to paginate using the next_token that comes in the response.

    # Here we can specify which fields we want to get in the response and the time range.

    # The header is set above with the bearer token. You can get the bearer token from your Twitter developer account.

    # To get a more detailed response, you can see the documentation here:
    # https://docs.x.com/x-api/posts/search-recent-posts#search-recent-posts

    # Set up the query parameters
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": ",".join(twitter_fields),
        "start_time": start_time,  # should be YYYY-MM-DDTHH:mm:ssZ format. E.g., 2023-01-01T00:00:00Z
        "end_time": end_time,  # should be YYYY-MM-DDTHH:mm:ssZ format. E.g., 2023-01-01T00:00:00Z
    }
    if next_token:
        params["next_token"] = next_token

    tweets = requests.get(
        "https://api.x.com/2/tweets/search/recent",
        params=params,
        headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
    )
    return tweets


result = search_recent_tweets_historical(
    "from:nytimes",
    max_results=10,
    twitter_fields=["id", "text", "created_at"],
    start_time="2024-06-01T00:00:00Z",
    end_time="2024-06-05T00:00:00Z",
)
print(
    f"tweets from nytimes between 2024-06-01 and 2024-06-05: {json.dumps(result.json(), indent=2)}"
)

recent tweets from nytimes between 2024-06-01 and 2024-06-05: {
  "errors": [
    {
      "parameters": {
        "start_time": [
          "2024-06-01T00:00Z"
        ]
      },
      "message": "Invalid 'start_time':'2024-06-01T00:00Z'. 'start_time' must be on or after 2025-10-15T18:20Z"
    },
    {
      "parameters": {
        "end_time": [
          "2024-06-05T00:00Z"
        ]
      },
      "message": "Invalid 'end_time':'2024-06-05T00:00Z'. 'end_time' must be on or after 2025-10-15T18:20Z"
    }
  ],
  "title": "Invalid Request",
  "detail": "One or more parameters to your request was invalid.",
  "type": "https://api.twitter.com/2/problems/invalid-request"
}


**OC19: Is the number of requests allowed by the API sufficient for monitoring more than 10,000 publications in 24 hours?**

*This item verifies whether data can be extracted without interruption and losses through the platform’s API for requests that accumulate more than 10,000 publications in 24 hours. The assessment should test the API to confirm that this volume of data can be collected continuously.*


In [ ]:
data_dir = "../../data"  # SET PATH
stats_filename = f"twitter-UGC-question-19-data-2025-10-24 11:16:55.758443"  # SET CORRECT FILENAME

# Create Path object
data = []
with open(f"{data_dir}/{stats_filename}", "r") as fin:
    for line in fin:
        data.append(json.loads(line))

print("Total Collected Data:", len(data))

display(data)

Total Collected Data: 20


[{'text': 'RT @g1: Donos de autoescola estacionam 200 carros na Ponte Estaiada, em SP, em protesto contra mudanças na CNH https://t.co/t4elbRyRYt #g1…',
  'edit_history_tweet_ids': ['1981726537784201283'],
  'id': '1981726537784201283'},
 {'text': 'Peraltro, in cambio di pochi punti % di energia globale, i quantitativi dei materiali impiegati per disseminare il globo di pannelli fotovoltaici e di turbine eoliche sono inimmaginabili. Scendendo in dettaglio, le stime IRENA ci dicono che, negli ultimi 35 anni, per i pannelli⤵️',
  'edit_history_tweet_ids': ['1981726527323504754'],
  'id': '1981726527323504754'},
 {'text': '@eldesmarque Otra basura mierdidista?? Lo más determinante de esta asociación criminal son los árbitros y los saben en cada rincón del globo. Mamporreros de mierda.',
  'edit_history_tweet_ids': ['1981726507937501557'],
  'id': '1981726507937501557'},
 {'text': 'RT @g1: Trump mira a Venezuela para combater narcotráfico, mas droga que mais causa overdoses nos EUA vem do 

### CONSISTENCY

**OC20: Are the results returned by the API consistently reproducible?**

*This item verifies whether data extracted via the platform’s API at any given time is consistent with other collections performed similarly, including content that has been deleted between collections. The assessment should conduct repeated test queries to confirm the reproducibility of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*

Test instructions: Please, develop a test that collects data for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [ ]:
# Use this cell, or more, to develop a process that collects post data more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:

import time


def search_recent_tweets(query, max_results=10, next_token=None):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params = {
        "query": query,
        "max_results": max_results,
        "next_token": next_token,
    }
    response = requests.get(url, headers=headers, params=params)
    return response


total_runs = 5
for idx in range(total_runs):
    index = idx + 1
    data_dir = "../../data"  # SET PATH
    filename = f"x-twitter-UGC-question-20-run-{index}-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S-%f')}.json"

    # DEVELOP YOUR CODE HERE
    response = search_recent_tweets("from:globo", max_results=10).json()
    with open(f"{data_dir}/{filename}", "w") as fout:
        json.dump(response, fout)
    time.sleep(15 * 60)  # Sleep for 15 minutes to avoid rate limits

In [ ]:
data_dir = "../../data"  # SET PATH

# Check if all data in different files are the same
file_pattern = "x-twitter-UGC-question-20-run-"

# Retrieve all files that match the pattern
import glob

file_list = glob.glob(f"{data_dir}/{file_pattern}*.json")

# Create Path object
data = []
for file_path in file_list:
    with open(file_path, "r") as fin:
        data.extend([tweet["id"] for tweet in json.load(fin)["data"]])
        # data.append(json.load(fin))

print(data)

print("Total Collected Data:", len(data))
print("Total Collected Unique Data:", len(set(data)))

# display(data)

['1983603131654222055', '1983603064411242989', '1983603007003799754', '1983602276410519964', '1983602189932081601', '1983602122395701556', '1983602023779229758', '1983601947501592952', '1983597734029631998', '1983597729801482648', '1983603131654222055', '1983603064411242989', '1983603007003799754', '1983602276410519964', '1983602189932081601', '1983602122395701556', '1983602023779229758', '1983601947501592952', '1983597734029631998', '1983597729801482648', '1983603131654222055', '1983603064411242989', '1983603007003799754', '1983602276410519964', '1983602189932081601', '1983602122395701556', '1983602023779229758', '1983601947501592952', '1983597734029631998', '1983597729801482648', '1983603131654222055', '1983603064411242989', '1983603007003799754', '1983602276410519964', '1983602189932081601', '1983602122395701556', '1983602023779229758', '1983601947501592952', '1983597734029631998', '1983597729801482648', '1983603131654222055', '1983603064411242989', '1983603007003799754', '198360227

**OC21: Is the data returned by the platform’s API consistent with the parameters and filters used in the request?**

*This item verifies whether the data extracted through the API accurately reflects the parameters and filters specified in the request. The assessment should conduct repeated test queries to confirm the consistency of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [5]:
# Use this cell, or more, to develop a process that collects post data more than one time using different parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
import time


def search_recent_tweets(
    query, params={"max_results": 10, "next_token": None}
):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params["query"] = query
    response = requests.get(url, headers=headers, params=params)
    return response


twitter_fields = [
    "id",
    "text",
    "created_at",
    "author_id",
    "lang",
    "public_metrics",
]

parameters = {
    # "start_time": "2026-02-04T00:00:00.000Z",
    # "end_time": "2026-02-04T00:00:00.000Z",
    "until_id": "2019080151649178006",
    # "max_results": 15,
    # "sort_order": "recency",

}  # SET A LIST WITH A LEAST 5 PARAMETERS
for idx, (key, param) in enumerate(parameters.items()):
    index = idx + 1
    data_dir = "../../data"  # SET PATH
    filename = f"x-twitter-UGC-question-21-run-{index}-{key}-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S-%f')}.json"

    # DEVELOP YOUR CODE HERE
    response = search_recent_tweets("from:globo", {key: param, "tweet.fields": ",".join(twitter_fields)}).json()

    with open(f"{data_dir}/{filename}", "w") as fout:
        json.dump(response, fout)
    time.sleep(60*20)  # To avoid hitting rate limits

In [6]:
data_dir = "../../data"  # SET PATH

# Check if all data in different files are the same
file_pattern = "x-twitter-UGC-question-21-run-"

# Retrieve all files that match the pattern
import glob

file_list = glob.glob(f"{data_dir}/{file_pattern}*.json")

# Create Path object

for file_path in file_list:
    with open(file_path, "r") as fin:
        # data.extend([tweet['id'] for tweet in json.load(fin)['data']])
        data = json.load(fin)
        print(f"\nData from file {file_path}:\n")
        if "errors" in data:
            print(f"Error found: {data['errors']}")
        else:
            # Check for consistency in the data
            if "data" in data:
                for tweet in data["data"]:
                    print(f"Tweet ID: {tweet['id']}, Text: {tweet['text']}")


Data from file ../../data/x-twitter-UGC-question-21-run-2-end_time-2025-10-30T15:51:12.804601.json:

Tweet ID: 1982902294871249075, Text: @tvglobo Se não fosse o apresentador, o @LucianoHuck estaria confirmado no próximo #DançaDosFamosos depois dessa performance! 🤣👏
Tweet ID: 1982902185102057799, Text: @tvglobo Mal podemos esperar pra assistir esse trio juntinho hoje! 😍
Tweet ID: 1982902086691107297, Text: @tvglobo O coração já tá acelerado só de pensar no que vem por aí... 😮
Tweet ID: 1982901999592190172, Text: @gshow Leonardo beeem discreto, né? 😅
Tweet ID: 1982901891106521211, Text: @gshow Essas duas mexem com o nosso coração! 🥺💙
Tweet ID: 1982901762144256192, Text: @gshow Essa semana promete fortes emoções, hein? 🤯
Tweet ID: 1982901642686280018, Text: @globoplay 🤣🤣🤣
Tweet ID: 1982901500121883112, Text: @geglobo O coração vibra de felicidade com essa conquista! 💙🇧🇷
Tweet ID: 1982901366097129667, Text: @novelagloboplay Avisa que não tem mais desculpa pra não acompanhar esse novelão!

### RELEVANCE

**OC22: Does the data extracted by the platform’s API reflect what is displayed on its user interface?**

*This item verifies whether the data returned by the API corresponds to the information displayed on the platform’s user interface at all levels of detail. The assessment should compare API responses with the user interface to confirm that key elements—such as authorship, complete content, interaction counts (e.g., comments, shares, replies), and referenced content (e.g., shares, mentions)—are fully represented.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# Compare API response fields with what is shown on the UI for a single tweet.
# The code requests the tweet with expanded author info and common fields that
# map to UI elements (authorship, full text, interaction counts, references).
# Replace tweet_id if you want to check a different tweet.

tweet_id = (
    tweet.get("id")
    if isinstance(tweet, dict) and "id" in tweet
    else "1979235854317920716"
)

url = f"https://api.x.com/2/tweets/{tweet_id}"
params = {
    "tweet.fields": ",".join(
        [
            "id",
            "text",
            "author_id",
            "created_at",
            "public_metrics",
            "conversation_id",
            "referenced_tweets",
            "entities",
        ]
    ),
    "expansions": "author_id",
    "user.fields": ",".join(
        ["id", "name", "username", "verified", "public_metrics", "location"]
    ),
}

response = requests.get(
    url,
    headers={"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"},
    params=params,
)

# Print raw response (useful when debugging rate limits or errors)
print("API status:", response.status_code)
try:
    resp_json = response.json()
except Exception:
    print("Failed to parse JSON response")
    resp_json = {}

print(json.dumps(resp_json, indent=2))

# If successful, extract key fields and map them to UI elements
if response.status_code == 200 and "data" in resp_json:
    tweet_data = resp_json["data"]
    users = {
        u["id"]: u for u in resp_json.get("includes", {}).get("users", [])
    }

    author = users.get(tweet_data.get("author_id"), {})
    public_metrics = tweet_data.get("public_metrics", {})

    print("\n--- MAPPED FIELDS ---")
    print(
        f"Authorship (UI: display name, handle): {author.get('name')} (@{author.get('username')})"
    )
    print(
        f"Author metadata (UI): verified={author.get('verified')}, location={author.get('location')}"
    )
    print(f"Full content (UI):\n{tweet_data.get('text')}")
    print(f"Created at (UI timestamp): {tweet_data.get('created_at')}")
    print(
        f"Interaction counts (UI badges): replies={public_metrics.get('reply_count')}, "
        f"retweets={public_metrics.get('retweet_count')}, likes={public_metrics.get('like_count')}, "
        f"quotes={public_metrics.get('quote_count')}"
    )
    print(
        f"Conversation id (used to fetch replies shown in UI): {tweet_data.get('conversation_id')}"
    )
    print(
        f"Referenced tweets (UI shows e.g., retweets/replies/quotes): {tweet_data.get('referenced_tweets')}"
    )
    print(
        f"Entities (mentions, urls, hashtags shown in UI): {tweet_data.get('entities')}"
    )

    # Quick checks to help assess completeness vs UI:
    checks = {
        "has_full_text": bool(tweet_data.get("text")),
        "has_author_info": bool(author),
        "has_public_metrics": bool(public_metrics),
        "has_references_or_entities": bool(
            tweet_data.get("referenced_tweets") or tweet_data.get("entities")
        ),
    }
    print("\nQuick completeness checks:", checks)
else:
    print("\nCould not retrieve tweet details (check rate limits or token).")

API status: 200
{
  "data": {
    "public_metrics": {
      "retweet_count": 0,
      "reply_count": 0,
      "like_count": 0,
      "quote_count": 0,
      "bookmark_count": 0,
      "impression_count": 38
    },
    "author_id": "67411820",
    "referenced_tweets": [
      {
        "type": "replied_to",
        "id": "1982902220073931106"
      }
    ],
    "text": "@GloboFilmes Viva o grande Maur\u00edcio de Sousa! \ud83d\udc99",
    "created_at": "2025-10-28T15:52:10.000Z",
    "edit_history_tweet_ids": [
      "1983200123598569678"
    ],
    "id": "1983200123598569678",
    "entities": {
      "annotations": [
        {
          "start": 27,
          "end": 43,
          "probability": 0.6376,
          "type": "Person",
          "normalized_text": "Maur\u00edcio de Sousa"
        }
      ],
      "mentions": [
        {
          "start": 0,
          "end": 12,
          "username": "GloboFilmes",
          "id": "67395999"
        }
      ]
    },
    "conversation_id": "1

The tweet data corresponds to what is displayed on the X (Twitter) user interface, including authorship, complete content, interaction counts, and referenced content at date of collection 31/10/2025.

**OC23: Does the platform’s API allow for filtering data based on publisher location?**

*This item verifies whether the API supports applying location-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that data on public posts can be filtered by publisher location.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

# There are geo-related fields in the Twitter API, but they are not consistently populated.
# You can pass on the query parameters to filter by location.
# The parameters are:
# place
# place_country
# has:geo (For tweets with geo info)
# point_radius (center point in lat,long and radius in km, e.g., "37.781157 -122.398720 10km")
# bounding_box (southwest_long,southwest_lat,northeast_long,northeast_lat)

# For the purpose of this example, I will search for tweets from a specific country (Brazil - BR). Since, to answer this question, only a single filter should be enough to demonstrate the functionality.

# Here we demonstrate that for the free tier of the Twitter API, it is not possible to filter tweets by location using the place_country parameter in the search query.
# The rules and filtering docs: https://docs.x.com/x-api/enterprise-gnip-2.0/fundamentals/rules-filtering#getting-started-with-enterprise-rules-and-queries-

location = "BR"


def search_recent_tweets(
    query, params={"max_results": 10, "next_token": None}
):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params["query"] = query
    response = requests.get(url, headers=headers, params=params)
    return response


result = search_recent_tweets(
    f"trump place_country:{location}",
    {
        "max_results": 10,
        "tweet.fields": ",".join(["id", "text", "created_at", "geo"]),
    },
)
print(
    f"recent tweets response for location '{location}': {json.dumps(result.json(), indent=2)}"
)

recent tweets response for location 'BR': {
  "errors": [
    {
      "parameters": {
        "query": [
          "trump place_country:BR"
        ]
      },
      "message": "There were errors processing your request: Reference to invalid operator 'place_country'. Operator is not available in current product or product packaging. Please refer to complete available operator list at http://t.co/operators. (at position 7)"
    }
  ],
  "title": "Invalid Request",
  "detail": "One or more parameters to your request was invalid.",
  "type": "https://api.twitter.com/2/problems/invalid-request"
}


**OC24: Does the platform’s API allow for filtering data based on content language?**

*This item verifies whether the API allows for applying language-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by content language.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
language = "pt"  # SET language


def search_recent_tweets(
    query, params={"max_results": 10, "next_token": None}
):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params["query"] = query
    response = requests.get(url, headers=headers, params=params)
    return response


result = search_recent_tweets(
    f"trump lang:{language}",
    {
        "max_results": 10,
        "tweet.fields": ",".join(["id", "text", "created_at", "geo"]),
    },
)
print(
    f"recent tweets response for language '{language}': {json.dumps(result.json(), indent=2)}"
)

recent tweets response for language 'pt': {
  "data": [
    {
      "edit_history_tweet_ids": [
        "1985124866194825450"
      ],
      "text": "RT @JornalDaCidadeO: Depois de decis\u00e3o inesperada de Moraes, advogado de Trump desafia: \"Pe\u00e7a ao Comando Vermelho tamb\u00e9m\" \nhttps://t.co/jye\u2026",
      "id": "1985124866194825450",
      "created_at": "2025-11-02T23:20:24.000Z"
    },
    {
      "edit_history_tweet_ids": [
        "1985124861300113567"
      ],
      "text": "RT @ASachsida: Temos que denunciar o genoc\u00eddio de crist\u00e3os na Nig\u00e9ria.\nhttps://t.co/agUVJwBQ4H",
      "id": "1985124861300113567",
      "created_at": "2025-11-02T23:20:23.000Z"
    },
    {
      "edit_history_tweet_ids": [
        "1985124850671694327"
      ],
      "text": "RT @VoxLiberdade: \ud83d\udea8 A ONU est\u00e1 t\u00e3o \"ocupada\" criticando, Bukele, Trump, Milei, e agora Claudio Castro no Brasil que esqueceu desses crist\u00e3o\u2026",
      "id": "1985124850671694

**OC25: Does the platform’s API allow for filtering data by specific time periods?**

*This item verifies whether the API allows applying temporal filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by custom time ranges.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
initial_date = "2025-10-28T00:00:00.000Z"
final_date = "2025-10-30T00:00:00.000Z"


parameters = {
    "start_time": initial_date,
    "end_time": final_date,
}


def search_recent_tweets(
    query, params={"max_results": 10, "next_token": None}
):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params["query"] = query
    response = requests.get(url, headers=headers, params=params)
    return response


result = search_recent_tweets(
    f"trump lang:{language}",
    {
        "max_results": 10,
        "tweet.fields": ",".join(["id", "text", "created_at", "geo"]),
        **parameters,
    },
)

# Check that the tweets are within the specified date range
if result.status_code == 200:
    tweets = result.json().get("data", [])
    for tweet in tweets:
        tweet_date = tweet["created_at"]
        if initial_date <= tweet_date <= final_date:
            print(f"Tweet {tweet['id']} is within the date range.")
        else:
            print(f"Tweet {tweet['id']} is outside the date range.")
else:
    print("Error fetching tweets.")

print(
    f"tweets response between {initial_date} and {final_date}: {json.dumps(result.json(), indent=2)}"
)

Tweet 1983685276053729671 is within the date range.
Tweet 1983685259817578610 is within the date range.
Tweet 1983685254998286486 is within the date range.
Tweet 1983685249839346174 is within the date range.
Tweet 1983685248996274241 is within the date range.
Tweet 1983685246756516252 is within the date range.
Tweet 1983685227655655484 is within the date range.
Tweet 1983685227064246435 is within the date range.
Tweet 1983685221515206685 is within the date range.
Tweet 1983685195451826239 is within the date range.
tweets response between 2025-10-28T00:00:00.000Z and 2025-10-30T00:00:00.000Z: {
  "data": [
    {
      "edit_history_tweet_ids": [
        "1983685276053729671"
      ],
      "id": "1983685276053729671",
      "text": "RT @LeoKasura: \ud83d\udea8\ud83c\uddfa\ud83c\uddf8URGENTE: Patriotas de Nova York est\u00e3o IMPLORANDO ao Presidente Donald Trump que \"REVOGUE a cidadania de Zohran Mamdani e\u2026",
      "created_at": "2025-10-29T23:59:59.000Z"
    },
    {
      "edit_

### TIMELINESS

**OC26: Can data from newly published content be extracted from the platform’s API in near real time?**

*This item verifies whether the API allows the collection of data from specific content within one hour of its publication. The assessment should test the endpoint for the main content type to confirm that it allows the ready extraction of recent public posts data.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# Print the most recent publication date find in the response.

collection_date = datetime.now().isoformat()


def search_recent_tweets(
    query, params={"max_results": 10, "next_token": None}
):
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {TWITTER_BEARER_TOKEN}"}
    params["query"] = query
    response = requests.get(url, headers=headers, params=params)
    return response


result = search_recent_tweets(
    f"trump lang:{language}",
    {
        "max_results": 10,
        "tweet.fields": ",".join(["id", "text", "created_at"]),
    },
)

# Get the most recent post date from the response
if result.status_code == 200:
    tweets = result.json().get("data", [])
    if tweets:
        most_recent_post_date = max(tweet["created_at"] for tweet in tweets)
    else:
        most_recent_post_date = None
else:
    most_recent_post_date = None

In [ ]:
from datetime import timezone, timedelta

# Difference between collection date and most recent post date
if most_recent_post_date:
    utc_minus_3 = timezone(timedelta(hours=-3))
    collection_dt = (
        datetime.fromisoformat(collection_date)
        .replace(tzinfo=utc_minus_3)
        .astimezone(timezone.utc)
    )
    most_recent_dt = datetime.fromisoformat(
        most_recent_post_date.replace("Z", "+00:00")
    )
    time_difference = collection_dt - most_recent_dt

    print("Collection date:", collection_dt.isoformat())
    print("Most recent post:", most_recent_dt.isoformat())
    print(
        "Time difference between collection date and most recent post date:",
        time_difference,
    )

Collection date: 2025-11-03T01:04:32.586611+00:00
Most recent post: 2025-11-03T01:04:01+00:00
Time difference between collection date and most recent post date: 0:00:31.586611
